# Pragmatic competence: full GPU experiment
Select **Runtime > Change runtime type > GPU**. Upload the supplied updated project ZIP; cloning GitHub alone would omit these local fixes. Free GPU availability is not guaranteed. Four training conditions and multiple test passes can exceed one free session. Results are written to Drive and the final cell exports a ZIP. Llama access must be approved on Hugging Face. No LLM results are fabricated if a cell fails.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, os
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/pragmatic-competence-llm")
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe ZIP path")
    z.extractall(ROOT)
os.chdir(ROOT)
print(ROOT)

In [ ]:
%pip install -r requirements-training.txt

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))
assert torch.cuda.get_device_properties(0).total_memory >= 14 * 1024**3, "Need at least 14 GiB GPU memory"
from huggingface_hub import notebook_login
notebook_login()  # Use a read token; do not paste it into saved code cells
subprocess.run([sys.executable, "src/run_pipeline.py", "prepare"], check=True)

## Execute research
`joint` is the essential first run. The next cell runs all four conditions by default. Reduce CONDITIONS to `joint` for a shorter session. Use the same frozen v2 benchmark throughout. Artifacts persist in Drive if Colab disconnects; re-running currently restarts the selected stages. Inspect training loss and saved predictions before making research claims.

In [ ]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
CONDITIONS = "joint,sarcasm,implicature,indirect"
subprocess.run([sys.executable, "src/run_pipeline.py", "llm", "--model", MODEL, "--conditions", CONDITIONS], check=True)
subprocess.run([sys.executable, "src/run_pipeline.py", "report"], check=True)

## Optional independent model baseline
Run only if sufficient runtime and storage remain; this downloads another 7B model.

In [ ]:
# subprocess.run([sys.executable, "src/evaluation/harness.py", "--model", "mistralai/Mistral-7B-Instruct-v0.3", "--output-dir", "results/llm/mistral"], check=True)

In [ ]:
import shutil
export = Path("/content/pragmatic-research-results.zip")
with zipfile.ZipFile(export, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["results", "experiments", "data/benchmark/v2"]:
        for p in (ROOT / folder).rglob("*"):
            if p.is_file() and not any(part.startswith("checkpoint-") for part in p.parts):
                z.write(p, p.relative_to(ROOT))
files.download(str(export))